# 02 — SmarTRIZ DPO Training

Loads preference pairs built by `scripts/build_dpo_pairs.py` and fine-tunes
the SFT model with DPO using the PEFT-no-ref + precompute strategy
(memory-safe on A100 40GB).

**Prerequisites:**
- Notebook 01 complete; `checkpoints/sft-7b/merged/` exists.
- `data/dpo_pairs.jsonl` exists (run the pair-builder cell below or
  `python scripts/build_dpo_pairs.py` outside the notebook).

In [1]:
# ─── CONFIG — edit this cell only ────────────────────────
import os, sys
from pathlib import Path

MODEL_SIZE = '7b'
DRIVE_PATH = '/content/drive/MyDrive/LIFTUP/smartriz/'

# Repo path on Colab — adjust if your repo lives elsewhere
REPO_PATH = '/content/smartriz-project'
if Path(REPO_PATH).exists() and REPO_PATH + '/src' not in sys.path:
    sys.path.insert(0, REPO_PATH + '/src')

# DeepInfra key kept for optional teacher fallback only (not required)
try:
    from google.colab import userdata
    DEEPINFRA_API_KEY = userdata.get('DEEPINFRA_API_KEY') or os.getenv('DEEPINFRA_API_KEY', '')
except Exception:
    DEEPINFRA_API_KEY = os.getenv('DEEPINFRA_API_KEY', '')

SFT_MODEL_DIR    = f'{DRIVE_PATH}checkpoints/sft-{MODEL_SIZE}/merged/'
OUTPUT_DIR       = f'{DRIVE_PATH}checkpoints/dpo-{MODEL_SIZE}/'
DPO_PAIRS_PATH   = f'{DRIVE_PATH}data/dpo_pairs.jsonl'
JUDGED_PATH      = f'{DRIVE_PATH}data/judged.jsonl'
VALIDATED_PATH   = f'{DRIVE_PATH}data/matrix_validated.jsonl'

# DPO training hyperparameters (per approved plan §3)
LORA_R              = 16
LORA_ALPHA          = 16              # alpha=r is the DPO standard
LORA_DROPOUT        = 0.05
MAX_LENGTH          = 2048
MAX_PROMPT_LENGTH   = 1024
BETA                = 0.1
LEARNING_RATE       = 1e-5            # adapter-tuned DPO; 5e-5 was 10x too high
NUM_EPOCHS          = 2
PER_DEVICE_BSZ      = 1               # 40 GB safety
GRAD_ACCUM_STEPS    = 8               # effective batch = 8
LOSS_TYPE           = 'apo_zero'      # SFT model underperforms winners; fallback: 'sigmoid'
WARMUP_RATIO        = 0.1
LR_SCHEDULER        = 'cosine'

# Runtime mode
REQUIRE_4BIT        = True   # True: fail fast if bitsandbytes CUDA is unavailable

# Pair-builder targets
PAIRS_TOTAL         = 3000
PAIRS_RATIOS        = '0.5,0.3,0.2'   # A:B:C — format/content/matrix

print(f'SFT source:    {SFT_MODEL_DIR}')
print(f'DPO output:    {OUTPUT_DIR}')
print(f'Pairs file:    {DPO_PAIRS_PATH}')
print(f'Loss / LR / β: {LOSS_TYPE} / {LEARNING_RATE} / {BETA}')


SFT source:    /content/drive/MyDrive/LIFTUP/smartriz/checkpoints/sft-7b/merged/
DPO output:    /content/drive/MyDrive/LIFTUP/smartriz/checkpoints/dpo-7b/
Pairs file:    /content/drive/MyDrive/LIFTUP/smartriz/data/dpo_pairs.jsonl
Loss / LR / β: apo_zero / 1e-05 / 0.1


In [2]:
# trl >= 0.11.4 needed for apo_zero + PEFT-no-ref + precompute_ref_log_probs
# Clean reinstall avoids stale CPU-only bitsandbytes wheels in Colab sessions.
# triton is pinned to 2.0.0 because triton>=2.1 removed triton.ops which
# bitsandbytes 0.45.x still imports unconditionally in triton_based_modules.py.
!pip uninstall -y -q bitsandbytes triton
!pip install -q --no-cache-dir transformers==4.45.2 peft==0.13.2 trl==0.11.4 \
  bitsandbytes==0.45.0 accelerate==0.34.2 datasets==3.0.1 wandb tqdm
!pip install -q --no-cache-dir triton==2.0.0

# Quick health check: bitsandbytes CUDA backend must be loaded for 4-bit.
import torch, importlib.metadata as md
print('torch.cuda.is_available =', torch.cuda.is_available(), '| torch.version.cuda =', torch.version.cuda)
print('bitsandbytes version =', md.version('bitsandbytes'))
print('triton version        =', md.version('triton'))
try:
    from bitsandbytes.cextension import lib as _bnb_lib
    print('bitsandbytes CUDA backend loaded =', _bnb_lib is not None)
except Exception as e:
    print('bitsandbytes import failed:', e)

# ⚠️ Restart runtime once after this cell, then run again from Cell 1.
# (Runtime → Restart session)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 186.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 399.5 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement triton==2.0.0 (from versions: 2.2.0, 2.3.0, 2.3.1, 3.0.0, 3.1.0, 3.2.0, 3.3.0, 3.3.1, 3.4.0, 3.5.0, 3.5.1, 3.6.0, 3.7.0)
ERROR: No matching distribution found for triton==2.0.0


torch.cuda.is_available = True | torch.version.cuda = 12.8
bitsandbytes version = 0.45.0
triton version        = 3.6.0
bitsandbytes import failed: No module named 'triton.ops'


In [3]:
import os
# Force uninstall and reinstall of bitsandbytes to fix CUDA backend loading issues
!pip uninstall -y bitsandbytes
!pip install --no-cache-dir bitsandbytes==0.45.0

Found existing installation: bitsandbytes 0.45.0
Uninstalling bitsandbytes-0.45.0:
  Successfully uninstalled bitsandbytes-0.45.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 298.2 MB/s eta 0:00:00


In [10]:
import os
import glob

# Find the actual bitsandbytes cuda library path
bnb_path = os.path.dirname(glob.glob('/usr/local/lib/python*/dist-packages/bitsandbytes/libbitsandbytes_cuda*.so')[0])
cuda_lib = glob.glob(f'{bnb_path}/libbitsandbytes_cuda*.so')[0]

# Force bitsandbytes to use the specific CUDA library found
os.environ['LD_LIBRARY_PATH'] = f'{bnb_path}:' + os.environ.get('LD_LIBRARY_PATH', '')
print(f'Found and linked bitsandbytes CUDA binary: {cuda_lib}')

Found and linked bitsandbytes CUDA binary: /usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda126.so


In [14]:
import bitsandbytes as bnb
import os
import ctypes
import glob

# Manually find the CUDA library
bnb_cuda_libs = glob.glob('/usr/local/lib/python*/dist-packages/bitsandbytes/libbitsandbytes_cuda*.so')
if bnb_cuda_libs:
    cuda_lib_path = bnb_cuda_libs[0]
    # In bitsandbytes 0.45.0, the library is typically handled in bnb.library.lib
    # We force-load the CUDA library into the internal reference
    try:
        bnb.library.lib = ctypes.cdll.LoadLibrary(cuda_lib_path)
        print(f'Successfully forced bitsandbytes to load: {cuda_lib_path}')
    except AttributeError:
        # Fallback for different internal structures
        from bitsandbytes.cextension import lib as _lib
        _lib = ctypes.cdll.LoadLibrary(cuda_lib_path)
        print(f'Loaded via cextension fallback: {cuda_lib_path}')
else:
    print('Could not find bitsandbytes CUDA binary.')

Loaded via cextension fallback: /usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda126.so


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# ── Clone repo so smartriz helpers + scripts are importable ──────────────
# Required because Cell 1 set REPO_PATH but the actual code lives in your
# GitHub repo, not in Drive. Adjust REPO_URL below to match your fork.
#   - Public repo:  https://github.com/<user>/smartriz-project.git
#   - Private repo: https://<TOKEN>@github.com/<user>/smartriz-project.git
import sys
from pathlib import Path

REPO_URL = 'https://github.com/sevketugurel/smartriz-project.git'

if not Path(REPO_PATH).exists():
    print(f'Cloning {REPO_URL} → {REPO_PATH}')
    !git clone --depth 1 {REPO_URL} {REPO_PATH}
else:
    print(f'Repo already present at {REPO_PATH} — pulling latest')
    !cd {REPO_PATH} && git pull --ff-only

if REPO_PATH + '/src' not in sys.path:
    sys.path.insert(0, REPO_PATH + '/src')

# Smoke test — fails loudly if the clone or PYTHONPATH is wrong
from smartriz.training.formatter import SYSTEM_PROMPT  # noqa: F401
from smartriz.eval.format_check import score_format_compliance  # noqa: F401
print('Repo helpers import OK')

Repo already present at /content/smartriz-project — pulling latest
Already up to date.
Repo helpers import OK


In [6]:
# ── Build DPO pairs (skips if file already exists) ───────────────────────
import json
from pathlib import Path

if Path(DPO_PAIRS_PATH).exists():
    n = sum(1 for _ in open(DPO_PAIRS_PATH))
    print(f'Pairs file already present: {n} pairs at {DPO_PAIRS_PATH}')
else:
    print(f'Building DPO pairs → {DPO_PAIRS_PATH}')
    !python {REPO_PATH}/scripts/build_dpo_pairs.py \
        --judged {JUDGED_PATH} \
        --validated {VALIDATED_PATH} \
        --out {DPO_PAIRS_PATH} \
        --total {PAIRS_TOTAL} \
        --ratios {PAIRS_RATIOS}

# Load
dpo_pairs = [json.loads(l) for l in open(DPO_PAIRS_PATH)]
print(f'Loaded {len(dpo_pairs)} preference pairs')

# Resume detection
checkpoint_dirs = sorted(Path(OUTPUT_DIR).glob('checkpoint-*')) \
    if Path(OUTPUT_DIR).exists() else []
RESUME_FROM = str(checkpoint_dirs[-1]) if checkpoint_dirs else None
print(f'DPO checkpoint: {RESUME_FROM or "none — training from scratch"}')


Pairs file already present: 2186 pairs at /content/drive/MyDrive/LIFTUP/smartriz/data/dpo_pairs.jsonl
Loaded 2186 preference pairs
DPO checkpoint: none — training from scratch


In [7]:
# ── Load SFT merged model + prepare for DPO ──
# Two cases handled automatically:
#   A) SFT model saved in 4-bit (bitsandbytes available) → load with original config
#   B) SFT model properly merged to bf16  (bitsandbytes absent) → strip config, load bf16
import torch
import sys
import types
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training

# --- ROBUST MONKEY PATCH FOR TRITON/BITSANDBYTES COMPATIBILITY ---
# Triton >= 2.1 removed 'triton.ops' and changed internal names like 'early_config_prune'.
try:
    import triton
    import triton.runtime

    # Fix for missing attributes in triton.runtime that bitsandbytes 0.45.0 expects
    for missing_attr in ['early_config_prune', 'estimate_matmul_time']:
        if not hasattr(triton.runtime, missing_attr):
            setattr(triton.runtime, missing_attr, lambda *args, **kwargs: None)

    if not hasattr(triton, 'ops'):
        # Create a mock module for triton.ops
        mock_ops = types.ModuleType('triton.ops')
        # Alias it to runtime so standard attributes are found
        for attr in dir(triton.runtime):
            setattr(mock_ops, attr, getattr(triton.runtime, attr))

        triton.ops = mock_ops
        sys.modules['triton.ops'] = mock_ops
        sys.modules['triton.ops.matmul_perf_model'] = mock_ops
except ImportError:
    pass
# -----------------------------------------------------------------

def _bnb_gpu_available():
    """Return True only if bitsandbytes CUDA library is loaded and functional."""
    if not torch.cuda.is_available():
        return False
    try:
        from bitsandbytes.cextension import lib as _bnb_lib
        return _bnb_lib is not None
    except Exception as e:
        print(f'[WARN] bitsandbytes CUDA check failed: {e}')
        return False

tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_bnb_ok = _bnb_gpu_available()

if REQUIRE_4BIT and not _bnb_ok:
    raise RuntimeError(
        'REQUIRE_4BIT=True but bitsandbytes CUDA backend is unavailable.\n'
        'Fix steps:\n'
        '1) Runtime -> Disconnect and delete runtime\n'
        '2) Ensure A100 GPU is selected\n'
        '3) Re-run Cell 2, then restart runtime once\n'
        '4) Re-run from Cell 1'
    )

if _bnb_ok:
    from transformers import BitsAndBytesConfig
    print('[INFO] bitsandbytes CUDA OK — loading SFT model in 4-bit (QLoRA DPO).')
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        SFT_MODEL_DIR,
        quantization_config=bnb_cfg,
        device_map='auto',
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
    print('[INFO] Model loaded in 4-bit + prepare_model_for_kbit_training applied.')
else:
    print('[INFO] bitsandbytes CUDA not available — loading SFT model in bf16.')
    sft_config = AutoConfig.from_pretrained(SFT_MODEL_DIR, trust_remote_code=True)
    if hasattr(sft_config, 'quantization_config'):
        delattr(sft_config, 'quantization_config')
    model = AutoModelForCausalLM.from_pretrained(
        SFT_MODEL_DIR,
        config=sft_config,
        device_map='auto',
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
    print('[INFO] Model loaded in bf16.')

model.config.use_cache = False
model.gradient_checkpointing_enable()
if hasattr(model, 'enable_input_require_grads'):
    model.enable_input_require_grads()

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules='all-linear',
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

total = sum(p.numel() for p in model.parameters())
print(f'Model loaded — {total:,} params.')
print('LoRA adapters will be applied by DPOTrainer via peft_config (PEFT-no-ref strategy).')

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


[INFO] bitsandbytes CUDA OK — loading SFT model in 4-bit (QLoRA DPO).


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:182: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[INFO] Model loaded in 4-bit + prepare_model_for_kbit_training applied.
Model loaded — 4,352,972,288 params.
LoRA adapters will be applied by DPOTrainer via peft_config (PEFT-no-ref strategy).


In [15]:
# ── Build HF dataset and run DPOTrainer ───────────────────────────────────
import sys
import os
import wandb
import glob
from datasets import Dataset as HFDataset
from trl import DPOTrainer, DPOConfig

# 1. Force CUDA backend for bitsandbytes manually to prevent CPU fallback error
try:
    bnb_path = os.path.dirname(glob.glob('/usr/local/lib/python*/dist-packages/bitsandbytes/libbitsandbytes_cuda*.so')[0])
    os.environ['LD_LIBRARY_PATH'] = f'{bnb_path}:' + os.environ.get('LD_LIBRARY_PATH', '')
except Exception:
    pass

# 2. Disable W&B logging and bypass the interactive init
USE_WANDB = False
if not USE_WANDB:
    os.environ['WANDB_MODE'] = 'disabled'
    print('[INFO] Weights & Biases disabled. Bypassing login.')

# Use the shared SmarTRIZ system prompt from the formatter module
sys.path.insert(0, REPO_PATH + '/src')
from smartriz.training.formatter import SYSTEM_PROMPT

# 3. Only initialize if explicitly requested to avoid hangs
if USE_WANDB:
    wandb.init(project='smartriz-dpo', name=f'dpo-{MODEL_SIZE}-{LOSS_TYPE}', resume='allow')

def format_dpo_prompt(pair):
    prompt = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user',   'content': pair['prompt']}],
        tokenize=False, add_generation_prompt=True
    )
    return {'prompt': prompt, 'chosen': pair['chosen'], 'rejected': pair['rejected']}

dpo_hf = HFDataset.from_list([format_dpo_prompt(p) for p in dpo_pairs])
print(f'DPO dataset size: {len(dpo_hf)}')

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=BETA,
    loss_type=LOSS_TYPE,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    per_device_train_batch_size=PER_DEVICE_BSZ,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    bf16=True,
    optim='adamw_torch_fused',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,
    precompute_ref_log_probs=True,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to='none',
    remove_unused_columns=False,
    run_name=f'dpo-{MODEL_SIZE}-{LOSS_TYPE}',
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_hf,
    tokenizer=tokenizer,
    peft_config=lora_cfg,
)

print('Starting DPO training (W&B disabled)...')
dpo_trainer.train(resume_from_checkpoint=RESUME_FROM)
print('DPO training complete')

[INFO] Weights & Biases disabled. Bypassing login.
DPO dataset size: 2186


Tokenizing train dataset:   0%|          | 0/2186 [00:00<?, ? examples/s]

Starting DPO training (W&B disabled)...


Train dataset reference log probs:   0%|          | 0/2186 [00:00<?, ?it/s]

AttributeError: /usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cpu.so: undefined symbol: cdequantize_blockwise_fp32

In [ ]:
# ── Merge DPO LoRA into base and save ─────────────────────────────────────
from pathlib import Path

merged_dpo = OUTPUT_DIR + 'merged/'
if not Path(merged_dpo + 'config.json').exists():
    print('Merging DPO LoRA weights...')
    merged = dpo_trainer.model.merge_and_unload()
    merged.save_pretrained(merged_dpo)
    tokenizer.save_pretrained(merged_dpo)
    print(f'DPO merged model saved to {merged_dpo}')
else:
    print(f'DPO merged model already exists at {merged_dpo}')

print()
print('Ready for Notebook 03 (GGUF conversion + 4-model evaluation)')
